<a href="https://colab.research.google.com/github/SANGHATI23/neuroimaging-fhir-omop-fidelity/blob/main/Phase49.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ICHI 2027 · Phase 49 — comparator-role amendment and executable synthetic results freeze

Phase 48 established two facts that change how the original four-arm plan must be reported:

1. frozen generic TFL v15 is reproducible in its own workflow;
2. the frozen runner has no generic new-resource event-construction API, so it cannot honestly be executed as a neuroimaging Arm C without changing the comparator.

Phase 49 records that feasibility-driven amendment transparently and freezes the **currently executable evidence** without changing the Phase 42/43 neuroimaging method or holdout truth.

### Amended evidence roles

- **A — faithful standard implementation:** retained as the confirmatory standards comparator, but deferred until the real source/vocabulary/MI-CDM gates clear.
- **B — value-only ablation:** retained as an executed synthetic stress-test ablation; it is not a conventional standard baseline.
- **C — frozen generic TFL v15:** retained as verified prior-art/reference evidence, not as an executed neuroimaging quantitative arm.
- **D — proposed neuroimaging TFL:** retained as the executed developer-authored synthetic method under the frozen Phase 42 rules and Phase 43 no-retuning holdout.

This is a **post-feasibility protocol amendment**, not a preregistration claim. It changes comparator role only. It does not modify the method, fault definitions, truth labels, or Phase 43 holdout.

### Outputs

The notebook produces:
- amended comparator-role table;
- protocol-amendment lock;
- Phase 42 incremental ablation table;
- Phase 43 holdout evidence summary;
- person-cluster bootstrap summaries for synthetic cohort consequences;
- B-versus-D synthetic comparison table;
- claim/evidence matrix for manuscript writing;
- two draft synthetic-only figures;
- public shareable summary and hashes.


In [1]:
# CODE CELL 1/9 — Drive, constants, helpers
import os, re, csv, json, hashlib, shutil, tempfile, zipfile, time, math, random
import datetime as dt
from pathlib import Path
from collections import defaultdict, Counter

PROJECT='ICHI2027_neuroimaging_fhir_omop_fidelity'
PHASE='49_comparator_role_amendment_and_executable_synthetic_results_freeze'
ANALYSIS_VERSION='phase49_amended_analysis_v1.0'
BOOTSTRAP_SEED=490930
BOOTSTRAP_REPS=5000

EXPECTED={
 'p48_summary':('ICHI2027_Phase48_SHAREABLE','.json','651a2492d3c00cb82f4abe4628fa22a9736cff3d7055848750c1a46f668841c2'),
 'p42_summary':('ICHI2027_Phase42_SHAREABLE','.json','fe20f91ced49f1d86f22cbdc44268ffbc078b6507c8442271d9fe7d73214f787'),
 'p42_ablation':('ICHI2027_Phase42_ABLATION_METRICS','.csv','194d507a1159bae926777bc12169a963dbaadd78a70bbf981b909edf5a7ffb1c'),
 'p42_rules':('ICHI2027_Phase42_RULE_COVERAGE','.csv','9b934320a0fd8c596cc9f425b52d37045acf28e28288c7143da016ab0145971a'),
 'p43_summary':('ICHI2027_Phase43_SHAREABLE','.json','c24b174761f1e18e92c4d5abf761f7ff5d7c5f09740c8a7e3d0f63a9af50495f'),
 'p43_cases':('ICHI2027_Phase43_HOLDOUT_CASES','.csv','cf8f5c1c8db0b26ea524c746a536046f19b4b3414b5c2b5049efc53acc1cef7c'),
 'p43_events':('ICHI2027_Phase43_HOLDOUT_EVENTS','.csv','f62992a75bbad73b36052327621d1176ff13bfd9887efc750feb80a4bff62c69'),
 'p43_cohort':('ICHI2027_Phase43_COHORT_CONSEQUENCES','.csv','43521bc017b319391c698bbc41804515aca0952dbfedcf66b6a3417829666d75'),
 'p47_applicability':('ICHI2027_Phase47_E0_E7_GENERIC_V15_APPLICABILITY','.csv','69aff9c4b7925b3868f557f6a503c1529950db22f993bf136f0d1a5348731060'),
}

def sha256(p):
    h=hashlib.sha256()
    with Path(p).open('rb') as f:
        for b in iter(lambda:f.read(2*1024*1024),b''):
            h.update(b)
    return h.hexdigest()

def write_csv(path,rows,fieldnames=None):
    rows=list(rows)
    if fieldnames is None:
        if not rows: raise ValueError('fieldnames required for empty CSV')
        fieldnames=list(rows[0].keys())
    with Path(path).open('w',newline='',encoding='utf-8') as f:
        w=csv.DictWriter(f,fieldnames=fieldnames)
        w.writeheader(); w.writerows(rows)

def qtile(vals,q):
    vals=sorted(float(x) for x in vals)
    if not vals: return float('nan')
    if len(vals)==1: return vals[0]
    pos=(len(vals)-1)*q
    lo=int(math.floor(pos)); hi=int(math.ceil(pos))
    if lo==hi:return vals[lo]
    return vals[lo]*(hi-pos)+vals[hi]*(pos-lo)

if os.environ.get('PHASE49_LOCAL_DRIVE_ROOT'):
    MYDRIVE=Path(os.environ['PHASE49_LOCAL_DRIVE_ROOT']).expanduser().resolve()
    ROOTS=[MYDRIVE]
    print('Offline Drive-mirror mode:',MYDRIVE)
else:
    from google.colab import drive
    drive.mount('/content/drive',force_remount=False)
    MYDRIVE=Path('/content/drive/MyDrive')
    assert MYDRIVE.is_dir()
    ROOTS=[MYDRIVE]
    sd=Path('/content/drive/Shareddrives')
    if sd.is_dir(): ROOTS.append(sd)

hint=os.environ.get('PHASE49_DRIVE_FOLDER_HINT','').strip()
if hint:
    hp=Path(hint).expanduser()
    if not hp.is_absolute(): hp=MYDRIVE/hp
    if not hp.is_dir(): raise FileNotFoundError(hp)
    ROOTS=[hp.resolve()]

WORK=Path(tempfile.mkdtemp(prefix='ichi2027_phase49_'))
STAGE=WORK/'outputs'; STAGE.mkdir()
OUT_BASE=Path(os.environ.get('PHASE49_OUTPUT_DIR',str(MYDRIVE/'ICHI2027'/'Phase49')))
OUT=OUT_BASE/dt.datetime.now(dt.timezone.utc).strftime('run_%Y%m%dT%H%M%S_%fZ')
AMENDMENT_LOCK=OUT_BASE/'ICHI2027_Phase49_PROTOCOL_AMENDMENT_LOCK.json'
print('Phase 49 output:',OUT)


Mounted at /content/drive
Phase 49 output: /content/drive/MyDrive/ICHI2027/Phase49/run_20261001T041542_454094Z


In [2]:
# CODE CELL 2/9 — Auto-discover and verify the exact evidence chain

def name_matches(actual,stem,ext):
    return re.fullmatch(re.escape(stem)+r'(?:-\d+|\s*\(\d+\))?'+re.escape(ext),actual,re.I) is not None

def find_by_sha(stem,ext,digest):
    good=[]
    for root in ROOTS:
        for here,dirs,files in os.walk(root,topdown=True,followlinks=False):
            dirs[:]=[d for d in dirs if d not in {'.git','node_modules','.ipynb_checkpoints'} and not d.startswith('.Trash')]
            for fn in files:
                if name_matches(fn,stem,ext):
                    p=Path(here)/fn
                    try:
                        if sha256(p)==digest: good.append(p)
                    except OSError: pass
    if not good:
        raise FileNotFoundError(f'No SHA-approved Drive copy found for {stem}{ext}')
    return sorted(good,key=str)[0]

P={key:find_by_sha(stem,ext,digest) for key,(stem,ext,digest) in EXPECTED.items()}

p48=json.loads(P['p48_summary'].read_text('utf-8'))
p42=json.loads(P['p42_summary'].read_text('utf-8'))
p43=json.loads(P['p43_summary'].read_text('utf-8'))

def read_csv(path):
    with Path(path).open(newline='',encoding='utf-8-sig') as f:
        return list(csv.DictReader(f))

abl42=read_csv(P['p42_ablation'])
rules42=read_csv(P['p42_rules'])
cases43=read_csv(P['p43_cases'])
events43=read_csv(P['p43_events'])
cohort43=read_csv(P['p43_cohort'])
c_scope=read_csv(P['p47_applicability'])

assert p48['phase']=='48_frozen_v15_bootstrap_only_replay_and_armC_executability_decision'
assert p48['native_v15_reproducible_in_own_workflow'] is True
assert p48['generic_event_constructor_found'] is False
assert p48['armC_neuroimaging_executed'] is False
assert p48['armA_remaining_gates_carried_forward']==10

assert p42['phase']=='42_internal_synthetic_policy_algorithm_and_ablations'
assert p42['method_version']=='phase42_internal_source_projection_v0.1'
assert len(abl42)==6 and len(rules42)==60

assert p43['phase']=='43_frozen_method_synthetic_holdout_no_retuning_audit'
assert p43['method_retuned_after_holdout_generation'] is False
assert len(cases43)==229 and len(events43)==240 and len(cohort43)==916
assert p43['independently_blinded_external_validation'] is False
assert len(c_scope)==8

print('PASS: Phase 42/43 synthetic chain + Phase 48 comparator feasibility evidence verified.')
print('Phase 42 method:',p42['method_version'])
print('Phase 43 holdout:',len(cases43),'cases /',len(events43),'sealed truth events')
print('Phase 48 C status:',p48['armC_execution_path_status'])


PASS: Phase 42/43 synthetic chain + Phase 48 comparator feasibility evidence verified.
Phase 42 method: phase42_internal_source_projection_v0.1
Phase 43 holdout: 229 cases / 240 sealed truth events
Phase 48 C status: NO_GENERIC_NEW_RESOURCE_EVENT_API_IN_FROZEN_RUNNER


In [3]:
# CODE CELL 3/9 — Freeze the comparator-role amendment

roles=[
 {
  'label':'A',
  'original_role':'faithful standard implementation',
  'original_definition':'Documented FHIR-to-OMOP mapping plus standards-supported imaging extension; retain natively supported fields.',
  'phase49_role':'DEFERRED_CONFIRMATORY_STANDARD_COMPARATOR',
  'quantitative_status_now':'NOT_EXECUTED',
  'reason':'Ten source/vocabulary/official-load gates remain open; no official OMOP/MI-CDM ETL has executed.',
  'allowed_use_now':'Protocol target and future confirmatory comparator only',
  'not_allowed_now':'No Arm A performance, cohort-equivalence, or superiority claims'
 },
 {
  'label':'B',
  'original_role':'value-only ablation',
  'original_definition':'Intentionally omit QA/provenance dimensions as a stress-test ablation.',
  'phase49_role':'EXECUTED_SYNTHETIC_STRESS_TEST',
  'quantitative_status_now':'EXECUTED_SYNTHETIC_ONLY',
  'reason':'Phase 42 value-only diagnostic and Phase 43 cohort diagnostic are available on developer-authored synthetic evidence.',
  'allowed_use_now':'Quantify causal consequence of intentionally losing context in the synthetic fixtures',
  'not_allowed_now':'Do not describe B as the conventional standard or production baseline'
 },
 {
  'label':'C',
  'original_role':'existing TFL baseline',
  'original_definition':'Deploy frozen generic TFL v15 without neuroimaging-specific rules.',
  'phase49_role':'VERIFIED_GENERIC_PRIOR_ART_REFERENCE',
  'quantitative_status_now':'NOT_EXECUTED_ON_NEUROIMAGING',
  'reason':'v15 reproduced in its own workflow, but frozen runner has no generic new-resource event API; forcing MRI support would modify the comparator.',
  'allowed_use_now':'Prior-art boundary, frozen warning/fate taxonomy, reproducibility reference',
  'not_allowed_now':'No numeric head-to-head C-versus-D detection result on MRI'
 },
 {
  'label':'D',
  'original_role':'proposed neuroimaging TFL',
  'original_definition':'Same base projection plus imaging invariants, provenance chain and explicit qualified/review outcomes.',
  'phase49_role':'EXECUTED_SYNTHETIC_PROPOSED_METHOD',
  'quantitative_status_now':'EXECUTED_DEVELOPER_AUTHORED_SYNTHETIC_ONLY',
  'reason':'Phase 42 rules were frozen and Phase 43 applied the unchanged method to a sealed developer-authored holdout.',
  'allowed_use_now':'Internal synthetic method/ablation/holdout evidence with explicit scope labels',
  'not_allowed_now':'No independent validation, real-MRI, clinical-safety, official-target, or production-generalization claim'
 }
]

amendment={
 'project':PROJECT,
 'analysis_version':ANALYSIS_VERSION,
 'phase':PHASE,
 'amendment_kind':'POST_FEASIBILITY_COMPARATOR_ROLE_AMENDMENT',
 'amendment_reason':'Frozen generic TFL v15 is reproducible in its own workflow but has no unchanged generic new-resource execution API for neuroimaging evidence.',
 'changes_method_rules':False,
 'changes_phase43_holdout_truth':False,
 'changes_phase43_holdout_seed':False,
 'changes_fault_definitions':False,
 'changes_primary_synthetic_B_D_inputs':False,
 'changes_only_comparator_role_reporting':True,
 'armA_deferred_not_removed':True,
 'armC_reclassified_to_prior_art_reference':True,
 'full_four_arm_claim_permitted':False,
 'bootstrap_seed':BOOTSTRAP_SEED,
 'bootstrap_reps':BOOTSTRAP_REPS,
 'input_hashes':{k:sha256(v) for k,v in P.items()}
}

amendment_blob=json.dumps(amendment,sort_keys=True,separators=(',',':')).encode('utf-8')
AMENDMENT_SHA=hashlib.sha256(amendment_blob).hexdigest()

if AMENDMENT_LOCK.exists():
    prior=json.loads(AMENDMENT_LOCK.read_text('utf-8'))
    assert prior['amendment_sha256']==AMENDMENT_SHA,(
        'A different Phase49 protocol-amendment lock already exists. Do not silently overwrite it.'
    )
    amendment_lock_origin='REUSED_EXISTING_LOCK'
else:
    OUT_BASE.mkdir(parents=True,exist_ok=True)
    lock_payload={'amendment_sha256':AMENDMENT_SHA,**amendment}
    tmp=WORK/'amendment_lock.json'
    tmp.write_text(json.dumps(lock_payload,sort_keys=True,indent=2)+'\n',encoding='utf-8')
    shutil.copyfile(tmp,AMENDMENT_LOCK)
    assert sha256(tmp)==sha256(AMENDMENT_LOCK)
    amendment_lock_origin='NEW_LOCK_CREATED'

print('Protocol amendment SHA-256:',AMENDMENT_SHA)
print('Lock:',amendment_lock_origin)
for r in roles:
    print(r['label'],r['phase49_role'],r['quantitative_status_now'])


Protocol amendment SHA-256: ab797f913e4afca4e2673aa6e2084bd7bce71cdc2c8a23b6af7b948df69827ed
Lock: NEW_LOCK_CREATED
A DEFERRED_CONFIRMATORY_STANDARD_COMPARATOR NOT_EXECUTED
B EXECUTED_SYNTHETIC_STRESS_TEST EXECUTED_SYNTHETIC_ONLY
C VERIFIED_GENERIC_PRIOR_ART_REFERENCE NOT_EXECUTED_ON_NEUROIMAGING
D EXECUTED_SYNTHETIC_PROPOSED_METHOD EXECUTED_DEVELOPER_AUTHORED_SYNTHETIC_ONLY


In [4]:
# CODE CELL 4/9 — Phase 42 incremental ablation contribution table

abl_by={r['ablation']:r for r in abl42}
order=['00_VALUE_ONLY_DIAGNOSTIC','01_UNIT_ANATOMY','02_PLUS_PAIRING','03_PLUS_LINEAGE','04_PLUS_TRACE','05_FULL']
assert list(abl_by.keys())==order or set(abl_by)==set(order)

incremental=[]
previous_matched=0
for name in order:
    r=abl_by[name]
    matched=int(r['internally_matched_fault_events'])
    injected=int(r['injected_fault_events'])
    increment=matched-previous_matched
    incremental.append({
      'ablation':name,
      'active_groups':r['active_groups'],
      'active_reason_codes':int(r['active_reason_codes']),
      'authored_fault_events':injected,
      'matched_authored_events':matched,
      'incremental_events_added_vs_previous_step':increment,
      'missed_authored_events':int(r['missed_injected_events']),
      'false_alert_events':int(r['false_alert_events']),
      'internal_event_recall':float(r['internal_event_recall']),
      'internal_event_precision':r['internal_event_precision'],
      'scope_warning':'DEVELOPER_AUTHORED_SYNTHETIC_NOT_INDEPENDENT_ESTIMATE'
    })
    previous_matched=matched

assert incremental[0]['matched_authored_events']==0
assert incremental[-1]['matched_authored_events']==195
assert incremental[-1]['false_alert_events']==0

# Full-rule family evidence.
full_families=[
 r for r in rules42 if r['ablation']=='05_FULL'
]
assert len(full_families)==10
assert sum(int(r['authored_truth_events']) for r in full_families)==195
assert all(int(r['missed_events'])==0 and int(r['false_alert_events'])==0 for r in full_families)

print('Incremental authored-event contributions:')
for r in incremental:
    print(r['ablation'],r['incremental_events_added_vs_previous_step'])


Incremental authored-event contributions:
00_VALUE_ONLY_DIAGNOSTIC 0
01_UNIT_ANATOMY 60
02_PLUS_PAIRING 45
03_PLUS_LINEAGE 45
04_PLUS_TRACE 15
05_FULL 30


In [5]:
# CODE CELL 5/9 — Phase 43 holdout evidence and cohort summaries

# Event summary by holdout kind / reason.
event_by_reason=defaultdict(lambda:{'truth':0,'matched':0,'missed':0,'false_alert':0})
for r in events43:
    a=event_by_reason[r['reason_code']]
    a['truth']+=int(r['sealed_truth'].lower()=='true')
    a['matched']+=int(r['matched'].lower()=='true')
    a['missed']+=int(r['missed'].lower()=='true')
    a['false_alert']+=int(r['false_alert'].lower()=='true')

holdout_event_rows=[]
for reason,a in sorted(event_by_reason.items()):
    holdout_event_rows.append({
      'reason_code':reason,
      'sealed_truth_events':a['truth'],
      'matched_events':a['matched'],
      'missed_events':a['missed'],
      'false_alert_events':a['false_alert'],
      'internal_match_fraction':a['matched']/a['truth'] if a['truth'] else '',
      'scope_warning':'SAME_PROJECT_AUTHORED_HOLDOUT_NOT_EXTERNAL_VALIDATION'
    })

assert sum(r['sealed_truth_events'] for r in holdout_event_rows)==240
assert sum(r['matched_events'] for r in holdout_event_rows)==240
assert sum(r['false_alert_events'] for r in holdout_event_rows)==0

# Cohort raw summaries by policy.
cohort_policy_rows=[]
by_policy=defaultdict(list)
for r in cohort43:
    by_policy[r['cohort_policy']].append(r)

for policy,rows in sorted(by_policy.items()):
    changed=sum(int(r['false_inclusions'])>0 or int(r['false_exclusions'])>0 for r in rows)
    j=[float(r['jaccard']) for r in rows]
    cohort_policy_rows.append({
      'cohort_policy':policy,
      'case_comparisons':len(rows),
      'cases_with_membership_change':changed,
      'membership_change_fraction':changed/len(rows),
      'total_false_inclusions_across_case_comparisons':sum(int(r['false_inclusions']) for r in rows),
      'total_false_exclusions_across_case_comparisons':sum(int(r['false_exclusions']) for r in rows),
      'mean_jaccard':sum(j)/len(j),
      'min_jaccard':min(j),
      'scope_warning':'SYNTHETIC_CASE_COMPARISONS_NOT_UNIQUE_PATIENT_ERROR_RATE'
    })

assert sum(r['case_comparisons'] for r in cohort_policy_rows)==916
print('Phase 43 cohort policy summaries:')
for r in cohort_policy_rows:
    print(r['cohort_policy'],r['cases_with_membership_change'],'/',r['case_comparisons'],'mean J',round(r['mean_jaccard'],4))


Phase 43 cohort policy summaries:
FROZEN_POLICY_PROXY 135 / 229 mean J 0.9607
FROZEN_POLICY_QUALIFIED 165 / 229 mean J 0.952
STRICT_PROJECTED_DIAGNOSTIC 120 / 229 mean J 0.9607
VALUE_ONLY_UNSAFE_DIAGNOSTIC 45 / 229 mean J 0.9869


In [6]:
# CODE CELL 6/9 — Person-cluster bootstrap and paired B-versus-D synthetic contrast

# The unit used here is the Phase43 case, clustered by its authored person_ref.
# This CI is an internal synthetic uncertainty summary, not a population-generalizable interval.

RNG=random.Random(BOOTSTRAP_SEED)
people=sorted({r['person_ref'] for r in cohort43})
assert len(people)==19

rows_by_policy_person=defaultdict(lambda:defaultdict(list))
for r in cohort43:
    rows_by_policy_person[r['cohort_policy']][r['person_ref']].append(r)

def summarize_rows(rows):
    if not rows:
        return {'change_fraction':float('nan'),'mean_jaccard':float('nan'),'mean_false_exclusions':float('nan')}
    changed=[int(int(r['false_inclusions'])>0 or int(r['false_exclusions'])>0) for r in rows]
    return {
      'change_fraction':sum(changed)/len(changed),
      'mean_jaccard':sum(float(r['jaccard']) for r in rows)/len(rows),
      'mean_false_exclusions':sum(int(r['false_exclusions']) for r in rows)/len(rows)
    }

bootstrap_rows=[]
for policy in sorted(by_policy):
    observed=summarize_rows(by_policy[policy])
    boot={'change_fraction':[],'mean_jaccard':[],'mean_false_exclusions':[]}
    pdata=rows_by_policy_person[policy]
    for _ in range(BOOTSTRAP_REPS):
        sample_people=[RNG.choice(people) for __ in people]
        sampled=[]
        for p in sample_people:
            sampled.extend(pdata.get(p,[]))
        s=summarize_rows(sampled)
        for k in boot:boot[k].append(s[k])
    bootstrap_rows.append({
      'cohort_policy':policy,
      'clusters':len(people),
      'bootstrap_reps':BOOTSTRAP_REPS,
      'observed_membership_change_fraction':observed['change_fraction'],
      'ci95_change_fraction_low':qtile(boot['change_fraction'],0.025),
      'ci95_change_fraction_high':qtile(boot['change_fraction'],0.975),
      'observed_mean_jaccard':observed['mean_jaccard'],
      'ci95_mean_jaccard_low':qtile(boot['mean_jaccard'],0.025),
      'ci95_mean_jaccard_high':qtile(boot['mean_jaccard'],0.975),
      'observed_mean_false_exclusions_per_case':observed['mean_false_exclusions'],
      'ci95_mean_false_exclusions_low':qtile(boot['mean_false_exclusions'],0.025),
      'ci95_mean_false_exclusions_high':qtile(boot['mean_false_exclusions'],0.975),
      'scope_warning':'INTERNAL_CLUSTER_BOOTSTRAP_ON_DEVELOPER_AUTHORED_SYNTHETIC_CASES'
    })

# Paired B-vs-D proxy: same case IDs.
B='VALUE_ONLY_UNSAFE_DIAGNOSTIC'
D='FROZEN_POLICY_QUALIFIED'
lookup=defaultdict(dict)
for r in cohort43:
    lookup[r['case_id']][r['cohort_policy']]=r

paired=[]
for cid,d in lookup.items():
    if B in d and D in d:
        rb,rd=d[B],d[D]
        assert rb['person_ref']==rd['person_ref']
        paired.append({
          'case_id':cid,
          'person_ref':rb['person_ref'],
          'B_change':int(int(rb['false_inclusions'])>0 or int(rb['false_exclusions'])>0),
          'D_change':int(int(rd['false_inclusions'])>0 or int(rd['false_exclusions'])>0),
          'B_jaccard':float(rb['jaccard']),
          'D_jaccard':float(rd['jaccard']),
          'B_false_exclusions':int(rb['false_exclusions']),
          'D_false_exclusions':int(rd['false_exclusions'])
        })

assert len(paired)==229
paired_by_person=defaultdict(list)
for r in paired:paired_by_person[r['person_ref']].append(r)

def paired_stats(rows):
    return {
      'change_fraction_difference_D_minus_B':sum(r['D_change']-r['B_change'] for r in rows)/len(rows),
      'mean_jaccard_difference_D_minus_B':sum(r['D_jaccard']-r['B_jaccard'] for r in rows)/len(rows),
      'mean_false_exclusion_difference_D_minus_B':sum(r['D_false_exclusions']-r['B_false_exclusions'] for r in rows)/len(rows)
    }

obs=paired_stats(paired)
boots={k:[] for k in obs}
for _ in range(BOOTSTRAP_REPS):
    sample_people=[RNG.choice(people) for __ in people]
    sampled=[]
    for p in sample_people:sampled.extend(paired_by_person.get(p,[]))
    s=paired_stats(sampled)
    for k,v in s.items():boots[k].append(v)

paired_summary=[{
 'comparison':'D_FROZEN_POLICY_QUALIFIED_MINUS_B_VALUE_ONLY_DIAGNOSTIC',
 'paired_cases':len(paired),
 'clusters':len(people),
 'bootstrap_reps':BOOTSTRAP_REPS,
 'change_fraction_difference':obs['change_fraction_difference_D_minus_B'],
 'change_fraction_difference_ci95_low':qtile(boots['change_fraction_difference_D_minus_B'],0.025),
 'change_fraction_difference_ci95_high':qtile(boots['change_fraction_difference_D_minus_B'],0.975),
 'mean_jaccard_difference':obs['mean_jaccard_difference_D_minus_B'],
 'mean_jaccard_difference_ci95_low':qtile(boots['mean_jaccard_difference_D_minus_B'],0.025),
 'mean_jaccard_difference_ci95_high':qtile(boots['mean_jaccard_difference_D_minus_B'],0.975),
 'mean_false_exclusion_difference':obs['mean_false_exclusion_difference_D_minus_B'],
 'mean_false_exclusion_difference_ci95_low':qtile(boots['mean_false_exclusion_difference_D_minus_B'],0.025),
 'mean_false_exclusion_difference_ci95_high':qtile(boots['mean_false_exclusion_difference_D_minus_B'],0.975),
 'interpretation_guardrail':'D policy intentionally withholds/reviews corrupted evidence; higher exclusion is a policy consequence, not automatically a performance loss.',
 'scope_warning':'SYNTHETIC_PROXY_COMPARISON_NOT_OFFICIAL_B_OR_D_TARGET_ETL'
}]

print('Person-cluster bootstrap complete:',BOOTSTRAP_REPS,'replicates.')
print('Paired B-D change fraction difference:',round(paired_summary[0]['change_fraction_difference'],4))


Person-cluster bootstrap complete: 5000 replicates.
Paired B-D change fraction difference: 0.524


In [7]:
# CODE CELL 7/9 — Claim/evidence matrix and manuscript-safe result statements

claim_rows=[
 {
  'claim_id':'C01',
  'claim':'The proposed neuroimaging rule set detected all authored Phase42 synthetic fault events with no false alerts in that development fixture set.',
  'evidence':'Phase42: 195/195 internally matched; 0 false alerts.',
  'status':'SUPPORTED_WITH_SCOPE',
  'mandatory_qualifier':'Developer-authored synthetic development evidence; not an independent sensitivity/PPV estimate.'
 },
 {
  'claim_id':'C02',
  'claim':'The unchanged frozen method matched all sealed authored Phase43 holdout fault events without unexpected alerts.',
  'evidence':'Phase43: 240/240 authored fault events matched; 0 unexpected alerts; truth sealed before detection; no retuning.',
  'status':'SUPPORTED_WITH_SCOPE',
  'mandatory_qualifier':'Same-project authored holdout; not independently blinded external validation.'
 },
 {
  'claim_id':'C03',
  'claim':'Adding imaging-specific rule groups incrementally increased authored-fault detection in the Phase42 ablation experiment.',
  'evidence':'Matched events progressed 0 → 60 → 105 → 150 → 165 → 195 across the frozen ablation sequence.',
  'status':'SUPPORTED_WITH_SCOPE',
  'mandatory_qualifier':'Internal synthetic ablation; increments reflect the designed rule families and fixture composition.'
 },
 {
  'claim_id':'C04',
  'claim':'Transformation faults and qualification policy can change synthetic cohort membership.',
  'evidence':'Phase43 person-level case comparisons show policy-dependent membership changes and Jaccard differences.',
  'status':'SUPPORTED_WITH_SCOPE',
  'mandatory_qualifier':'Synthetic proxy cohort; clinical MCI status not attested; not an official OMOP/MI-CDM target cohort.'
 },
 {
  'claim_id':'C05',
  'claim':'Frozen generic TFL v15 is reproducible in its own workflow.',
  'evidence':'Phase48 bootstrap-only replay passed with all non-bootstrap method-cell hashes preserved.',
  'status':'SUPPORTED',
  'mandatory_qualifier':'This establishes reproducibility of v15 in its own workflow, not neuroimaging Arm C execution.'
 },
 {
  'claim_id':'C06',
  'claim':'Frozen generic TFL v15 was quantitatively compared with the proposed neuroimaging method on the MRI fixtures.',
  'evidence':'No generic new-resource event-construction API exists in the frozen runner; neuroimaging Arm C was not executed.',
  'status':'NOT_SUPPORTED',
  'mandatory_qualifier':'Do not report C-versus-D quantitative performance.'
 },
 {
  'claim_id':'C07',
  'claim':'The proposed method outperforms a faithful standards-based OMOP/MI-CDM implementation.',
  'evidence':'Arm A remains blocked by 10 evidence/execution gates and official target ETL has not run.',
  'status':'NOT_SUPPORTED',
  'mandatory_qualifier':'Do not make standard-baseline superiority claims.'
 },
 {
  'claim_id':'C08',
  'claim':'The method is validated on real MRI/ADNI data or demonstrates clinical safety.',
  'evidence':'Real authorized MRI rows processed = 0; independent external validation = false.',
  'status':'NOT_SUPPORTED',
  'mandatory_qualifier':'Synthetic-only evidence cannot establish operational or clinical safety/generalizability.'
 }
]

safe_results=[
 {
  'result_id':'R01',
  'section':'Synthetic development ablation',
  'result_text':'Across 195 developer-authored fault events, the value-only diagnostic matched 0 events; successive rule groups increased matched events to 60, 105, 150, 165, and 195, with zero false alerts in this fixture set.',
  'scope':'INTERNAL_SYNTHETIC_ONLY'
 },
 {
  'result_id':'R02',
  'section':'Frozen synthetic holdout',
  'result_text':'With the Phase42 method unchanged, Phase43 matched all 240 sealed developer-authored holdout fault events, produced zero unexpected alerts, and correctly treated all 15 valid normalization controls without alert.',
  'scope':'SAME_PROJECT_AUTHORED_HOLDOUT_NOT_EXTERNAL_VALIDATION'
 },
 {
  'result_id':'R03',
  'section':'Comparator feasibility',
  'result_text':'Frozen generic TFL v15 replayed successfully in its original workflow, but its frozen runner exposed no generic new-resource event-construction API; it was therefore retained as prior-art/reference evidence rather than reported as an executed neuroimaging quantitative arm.',
  'scope':'EXECUTION_AND_PORTABILITY_EVIDENCE'
 },
 {
  'result_id':'R04',
  'section':'Standards comparator',
  'result_text':'The faithful standards-based Arm A remains deferred because the required source, vocabulary, and official-load evidence gates are incomplete.',
  'scope':'GATE_STATUS_NOT_PERFORMANCE_RESULT'
 }
]

print('Claim/evidence matrix:',len(claim_rows),'claims.')
print('Manuscript-safe result statements:',len(safe_results))


Claim/evidence matrix: 8 claims.
Manuscript-safe result statements: 4


In [8]:
# CODE CELL 8/9 — Draft synthetic-only figures

import matplotlib.pyplot as plt

# Figure A: ablation matched authored events.
fig1=STAGE/'ICHI2027_Phase49_FIGURE_ABLATION_INCREMENTAL.png'
labels=[r['ablation'].replace('00_','').replace('01_','').replace('02_','').replace('03_','').replace('04_','').replace('05_','') for r in incremental]
vals=[r['matched_authored_events'] for r in incremental]
plt.figure(figsize=(10,5.5))
plt.plot(range(len(labels)),vals,marker='o')
plt.xticks(range(len(labels)),labels,rotation=25,ha='right')
plt.ylabel('Matched authored fault events')
plt.xlabel('Frozen synthetic ablation')
plt.title('Internal synthetic ablation: cumulative authored-event detection')
plt.ylim(0,max(vals)*1.1)
plt.tight_layout()
plt.savefig(fig1,dpi=180)
plt.close()

# Figure B: cohort membership-change fraction by policy.
fig2=STAGE/'ICHI2027_Phase49_FIGURE_COHORT_POLICY.png'
policy_order=['STRICT_PROJECTED_DIAGNOSTIC','VALUE_ONLY_UNSAFE_DIAGNOSTIC','FROZEN_POLICY_PROXY','FROZEN_POLICY_QUALIFIED']
pmap={r['cohort_policy']:r for r in cohort_policy_rows}
xlabels=['Strict projected','Value-only','Frozen policy proxy','Frozen policy qualified']
y=[pmap[p]['membership_change_fraction'] for p in policy_order]
plt.figure(figsize=(9,5.5))
plt.bar(range(len(xlabels)),y)
plt.xticks(range(len(xlabels)),xlabels,rotation=20,ha='right')
plt.ylabel('Fraction of synthetic case comparisons with membership change')
plt.title('Synthetic cohort consequence varies by evidence policy')
plt.ylim(0,max(y)*1.2)
plt.tight_layout()
plt.savefig(fig2,dpi=180)
plt.close()

assert fig1.is_file() and fig2.is_file()
print('Draft figures generated:',fig1.name,',',fig2.name)


Draft figures generated: ICHI2027_Phase49_FIGURE_ABLATION_INCREMENTAL.png , ICHI2027_Phase49_FIGURE_COHORT_POLICY.png


In [9]:
# CODE CELL 9/9 — Save amendment/results package and verify Drive read-back

def save(name,rows,fields=None):
    p=STAGE/name
    write_csv(p,rows,fields)
    return p

artifacts=[]
artifacts.append(save('ICHI2027_Phase49_COMPARATOR_ROLES.csv',roles))
artifacts.append(save('ICHI2027_Phase49_ABLATION_INCREMENTAL.csv',incremental))
artifacts.append(save('ICHI2027_Phase49_HOLDOUT_EVENT_SUMMARY.csv',holdout_event_rows))
artifacts.append(save('ICHI2027_Phase49_COHORT_POLICY_SUMMARY.csv',cohort_policy_rows))
artifacts.append(save('ICHI2027_Phase49_CLUSTER_BOOTSTRAP.csv',bootstrap_rows))
artifacts.append(save('ICHI2027_Phase49_B_VS_D_SYNTHETIC_PAIRED.csv',paired_summary))
artifacts.append(save('ICHI2027_Phase49_CLAIM_EVIDENCE_MATRIX.csv',claim_rows))
artifacts.append(save('ICHI2027_Phase49_MANUSCRIPT_SAFE_RESULTS.csv',safe_results))
artifacts.append(save('ICHI2027_Phase49_C_PRIOR_ART_E0_E7_SCOPE.csv',c_scope))

amendment_path=STAGE/'ICHI2027_Phase49_PROTOCOL_AMENDMENT.json'
amendment_public={
 'amendment_sha256':AMENDMENT_SHA,
 'lock_origin':amendment_lock_origin,
 **amendment,
 'comparator_roles':roles
}
amendment_path.write_text(json.dumps(amendment_public,sort_keys=True,indent=2)+'\n',encoding='utf-8')
artifacts.append(amendment_path)

fig1=STAGE/'ICHI2027_Phase49_FIGURE_ABLATION_INCREMENTAL.png'
fig2=STAGE/'ICHI2027_Phase49_FIGURE_COHORT_POLICY.png'
artifacts.extend([fig1,fig2])

readme=STAGE/'ICHI2027_Phase49_README.txt'
readme.write_text(
 'Phase 49 is a transparent post-feasibility comparator-role amendment and synthetic-results freeze.\n'
 'It does not change the Phase42 neuroimaging rules, Phase43 holdout seed, holdout truth, or fault definitions.\n'
 'A remains a deferred confirmatory standards comparator; B remains a synthetic stress-test ablation; C is verified generic prior-art/reference evidence; D is the executed developer-authored synthetic proposed method.\n'
 'No full four-arm result is claimed.\n'
 'Cluster-bootstrap intervals summarize uncertainty inside the developer-authored synthetic case set and are not population-generalizable confidence intervals.\n'
 'Real MRI, independent external validation, official OMOP ETL, and official MI-CDM load remain unexecuted.\n',
 encoding='utf-8'
)
artifacts.append(readme)

summary={
 'project':PROJECT,
 'phase':PHASE,
 'analysis_version':ANALYSIS_VERSION,
 'run_utc':dt.datetime.now(dt.timezone.utc).isoformat(),
 'protocol_amendment_sha256':AMENDMENT_SHA,
 'protocol_amendment_kind':'POST_FEASIBILITY_COMPARATOR_ROLE_AMENDMENT',
 'method_rules_changed':False,
 'phase43_holdout_truth_changed':False,
 'arm_roles':{r['label']:r['phase49_role'] for r in roles},
 'armA_executed':False,
 'armB_synthetic_stress_test_executed':True,
 'armC_neuroimaging_executed':False,
 'armC_verified_prior_art_reference':True,
 'armD_developer_authored_synthetic_executed':True,
 'full_A_B_C_D_evaluation':False,
 'phase42_authored_fault_events':195,
 'phase42_full_rule_matched_events':195,
 'phase42_full_rule_false_alerts':0,
 'phase43_holdout_cases':229,
 'phase43_authored_fault_events':240,
 'phase43_matched_authored_fault_events':240,
 'phase43_unexpected_alert_events':0,
 'phase43_valid_normalizations':15,
 'phase43_independent_external_validation':False,
 'cluster_bootstrap_reps':BOOTSTRAP_REPS,
 'cluster_bootstrap_seed':BOOTSTRAP_SEED,
 'official_omop_etl_executed':False,
 'official_micdm_load_executed':False,
 'real_authorized_mri_processed':False,
 'independently_blinded_external_validation':False,
 'manuscript_claims_supported_with_scope':sum(r['status']=='SUPPORTED_WITH_SCOPE' for r in claim_rows),
 'manuscript_claims_supported':sum(r['status']=='SUPPORTED' for r in claim_rows),
 'manuscript_claims_not_supported':sum(r['status']=='NOT_SUPPORTED' for r in claim_rows),
 'input_sha256':{k:sha256(v) for k,v in P.items()},
 'artifact_sha256':{p.name:sha256(p) for p in artifacts},
 'warning':'This package freezes the currently executable synthetic evidence after a feasibility-driven comparator amendment. It is not a complete four-arm, official-target, real-MRI, or independently validated study.'
}
summary_path=STAGE/'ICHI2027_Phase49_SHAREABLE.json'
summary_path.write_text(json.dumps(summary,sort_keys=True,indent=2)+'\n',encoding='utf-8')
artifacts.append(summary_path)

zip_path=STAGE/'ICHI2027_Phase49_PUBLIC_OUTPUTS.zip'
with zipfile.ZipFile(zip_path,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for p in artifacts:z.write(p,p.name)

OUT.mkdir(parents=True,exist_ok=False)
for p in artifacts+[zip_path]:
    dest=OUT/p.name
    for attempt in range(3):
        try:
            shutil.copyfile(p,dest)
            assert sha256(dest)==sha256(p)
            break
        except Exception as e:
            if attempt==2: raise RuntimeError('Drive save/readback verification failed for '+p.name) from e
            time.sleep(1+attempt)

print('SUCCESS: Phase 49 amendment and synthetic-results freeze completed.')
print('Amendment SHA-256:',AMENDMENT_SHA)
print('Full four-arm evaluation: False')
print('Shareable:',OUT/summary_path.name)
print('Public ZIP:',OUT/zip_path.name)


SUCCESS: Phase 49 amendment and synthetic-results freeze completed.
Amendment SHA-256: ab797f913e4afca4e2673aa6e2084bd7bce71cdc2c8a23b6af7b948df69827ed
Full four-arm evaluation: False
Shareable: /content/drive/MyDrive/ICHI2027/Phase49/run_20261001T041542_454094Z/ICHI2027_Phase49_SHAREABLE.json
Public ZIP: /content/drive/MyDrive/ICHI2027/Phase49/run_20261001T041542_454094Z/ICHI2027_Phase49_PUBLIC_OUTPUTS.zip


## What Phase 49 changes — and what it does not

Phase 49 does **not** rescue missing experiments by renaming them.

It explicitly records that:
- Arm A still requires real evidence before execution;
- Arm C cannot be converted into an MRI comparator by adding new MRI rules;
- the currently quantitative evidence is synthetic B/D plus D's frozen holdout;
- the manuscript may report those synthetic results only with their scope limitations.

The next material scientific advance after this freeze is **external evidence**, not more self-authored synthetic optimization: an executable Arm A source/vocabulary path, authorized real MRI evaluation, or independently authored/blinded validation cases.
